# 02 — Building the CT-Level Census Tract Profile

This notebook rebuilds the census-tract (CT) level demographic, housing, and citizenship
feature table that feeds the rest of the `toronto_election_turnout` pipeline. It replaces
`analysis/archive/toronto_election_turnout/archive/census/scripts/build_ct_census_variables.py`
(840 lines) with a single narrated notebook that ports the same transformations.

**What this notebook produces:** one CT-level table (622 Toronto census tracts) with:
- **Block 1** (demographic): population, age structure, median age, household size, low income,
  education, unemployment.
- **Block 2** (housing): tenure, dwelling structure type (detached / semi / apartment), condo
  status, plus two density variables (`apartments_per_km2`, `condos_per_km2`).
- **Block 3** (immigration / citizenship / diversity): citizenship, immigrant status, visible
  minority, official language knowledge, mother tongue.
- A few Block 5 fields that happen to live in the same Census Profile row (subsidized housing
  share, transit commute share) — carried through here since they cost nothing extra, but they
  are Block 5's business downstream, not Block 2's.

**Simplification versus the old pipeline:** the archived pipeline computed `apartment_share`,
`condo_share`, and the raw dwelling-structure counts in `build_ct_census_variables.py`, but the
two *density* variables (`apartments_per_km2`, `condos_per_km2`) were only added later by an
undocumented "housing augmentation" step with no committed builder script (reconstructed only
from `housing_augmented_median_imputation_report.csv`). Since both densities need nothing more
than `land_area_km2` — already available on this same CT row — this notebook computes them here,
directly alongside the rest of Block 2. That folds the old bolt-on augmentation step into a single
clean Block-2 pass, with no separate notebook or script required.

**Out of scope here (per the project-level plan):** poll-to-CT interpolation, Block 4/5 joins,
median imputation, and the PLS model — those are notebooks 03–05.


## A reproducibility note, read before anything else

Two things needed by the archived script are **not available on this machine**, and this
notebook is honest about working around both rather than pretending to re-run steps that can't
actually run here.

**1. The two StatCan citizenship-extraction scripts are unrunnable here.**
`extract_statcan_census_profile_citizens_18plus.py` and `extract_statcan_population_18plus.py`
read from hardcoded `/private/tmp/statcan_da_ontario_ci.zip` / `/private/tmp/statcan_ct_ci.zip`
paths — leftovers from a different (likely macOS) machine. Those multi-GB comprehensive StatCan
downloads are not, and should not be, checked into this repo. Their **outputs are already
cached**, however, at:
- `data/toronto_election_turnout/archive/census/processed/ct/intermediate/statcan_2021_ct_citizens_18plus.csv`
- `data/toronto_election_turnout/archive/census/processed/ct/intermediate/statcan_2021_ct_population_18plus.csv`

This notebook loads those two CT-level files directly as its starting point for the
`canadian_citizens_18plus_count` / `population_18plus` fields. The matching **DA-level** files
(`.../da/intermediate/statcan_2021_da_citizens_18plus.csv` and
`.../da/intermediate/statcan_2021_da_population_18plus.csv`) exist alongside them but are **not**
used here — they're needed by notebook 03 for the DA-weighted poll→CT interpolation, not by this
CT-level profile.

**2. The ~35-characteristic Census Profile zip referenced by the archived script is also
missing — and unlike the citizenship case, there's no separate cached intermediate for it.**
`build_ct_census_variables.py` reads `census/raw/source_downloads/statcan_2021_ct_profile.zip`
(StatCan GEONO=007, the comprehensive Census Profile CSV) for population, dwelling structure,
tenure, condo status, low income, mobility, education, and citizenship counts. That zip is not on
this machine — it was checked for and confirmed absent (only the age-cube zips and boundary files
remain in `source_downloads/`). This notebook works around that the same way it works around the
citizenship zips: it treats the already-computed `statcan_2021_ct_census_variables_master.csv`
(the archived, ground-truth output of that same script, run on a machine that *did* have the zip)
as the **cached raw-characteristic extraction** — i.e. it reads the raw StatCan characteristic
values straight off that file, not its derived shares, and recomputes every derived share/count
independently in this notebook. A partial, independently-sourced cross-check below (from a sibling
project's own StatCan extract) confirms those cached raw values line up exactly wherever an
independent source is available, which is the best confidence check possible without the original
zip.

The upshot: the **raw extraction step** for ~35 Block 1–3 characteristics can't be independently
re-run here, but the **age-band construction** (from the raw age-cube zip, which *is* available),
the **population-18+/citizenship join logic**, all **~25 derived share variables**, and the two
new **Block 2 density variables** are all computed fresh in this notebook and are what the
verification cell at the end meaningfully tests.


In [1]:
import zipfile
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd


In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT = Path("../..").resolve()
DATA_ROOT = REPO_ROOT / "data" / "toronto_election_turnout"
ARCHIVE_ROOT = DATA_ROOT / "archive"
CENSUS_ROOT = ARCHIVE_ROOT / "census"
CENSUS_PROCESSED = CENSUS_ROOT / "processed"

# CT geometry universe (622 Toronto CTs) — also our ct_id/ctuid/dguid crosswalk.
CT_GEOMETRY_GEOJSON = CENSUS_PROCESSED / "ct" / "statcan_2021_toronto_ct.geojson"

# Single-year age cube (Table 98-10-0024-01) — fully available locally, used to build age bands.
CT_AGE_ZIP = CENSUS_ROOT / "raw" / "source_downloads" / "statcan_2021_ct_age_single_year_98100024-eng.zip"
CT_AGE_ZIP_MEMBER = "98100024.csv"

# Cached outputs of the two unrunnable citizenship-extraction scripts (see markdown note above).
CT_CITIZENS_18PLUS_INTERMEDIATE = CENSUS_PROCESSED / "ct" / "intermediate" / "statcan_2021_ct_citizens_18plus.csv"
CT_POPULATION_18PLUS_INTERMEDIATE = CENSUS_PROCESSED / "ct" / "intermediate" / "statcan_2021_ct_population_18plus.csv"
# DA-level counterparts, left alone here — notebook 03 needs these for interpolation weights, not us.
DA_CITIZENS_18PLUS_INTERMEDIATE = CENSUS_PROCESSED / "da" / "intermediate" / "statcan_2021_da_citizens_18plus.csv"
DA_POPULATION_18PLUS_INTERMEDIATE = CENSUS_PROCESSED / "da" / "intermediate" / "statcan_2021_da_population_18plus.csv"

# Old pipeline's interpolation outputs (untouched sibling tree) — used only to flag which CTs
# fall inside the 585-CT interpolation universe, exactly as the archived script did.
INTERPOLATION_PROCESSED = ARCHIVE_ROOT / "interpolation" / "processed"

# The Census Profile zip the archived script reads — confirmed NOT present on this machine.
CT_PROFILE_ZIP = CENSUS_ROOT / "raw" / "source_downloads" / "statcan_2021_ct_profile.zip"
# The archived, ground-truth master output — doubles as (a) our stand-in cached raw-characteristic
# source (see markdown note above) and (b) the verification target at the end of this notebook.
CT_CENSUS_VARIABLES_MASTER = CENSUS_PROCESSED / "ct" / "statcan_2021_ct_census_variables_master.csv"
# An independent partial extract of the same underlying StatCan product, built for a different
# analysis in this repo — used only as a bonus cross-check, not as a primary source.
CT_SMALL_PROFILE_CROSSCHECK = REPO_ROOT / "data" / "clustering_neighbourhoods" / "census" / "toronto-ct-small.csv"
# Downstream reference for the two new density variables, which have no home in the master file.
HOUSING_AUGMENTED_REFERENCE = (
    ARCHIVE_ROOT / "modelling" / "processed" / "spatial_models"
    / "toronto_ct_blocks_1_5_model_input_housing_augmented_median_imputed.csv"
)

# ── Outputs ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = DATA_ROOT / "census"
OUTPUT_CSV = OUTPUT_DIR / "ct_census_profile.csv"
OUTPUT_GEOJSON = OUTPUT_DIR / "ct_census_profile.geojson"

# ── 2021 Census Profile characteristic IDs -> field names (Blocks 1-3) ─────────
# Identical mapping to the archived build script; documents provenance for every raw field below.
PROFILE_CHARACTERISTICS = {
    "population_total": 1,
    "population_density_per_km2": 6,
    "land_area_km2": 7,
    "median_age": 40,
    "structural_type_total_dwellings": 41,
    "single_detached_house_count": 42,
    "semi_detached_house_count": 43,
    "apartment_duplex_count": 45,
    "apartment_lt5_storeys_count": 46,
    "apartment_5plus_storeys_count": 47,
    "average_household_size": 57,
    "low_income_status_total": 335,
    "low_income_lim_at_count": 340,
    "low_income_lim_at_prevalence_pct_official": 345,
    "official_language_knowledge_total": 383,
    "neither_english_nor_french_count": 387,
    "mother_tongue_total": 393,
    "non_official_mother_tongue_count": 398,
    "tenure_total_households": 1414,
    "owner_households_count": 1415,
    "renter_households_count": 1416,
    "condo_status_total_dwellings": 1418,
    "condominium_dwellings_count": 1419,
    "subsidized_housing_tenant_pct_official": 1491,
    "citizenship_total": 1522,
    "canadian_citizens_18plus_count": 1525,
    "not_canadian_citizens_count": 1526,
    "immigrant_status_total": 1527,
    "immigrants_count": 1529,
    "recent_immigrants_2016_2021_count": 1536,
    "visible_minority_total": 1683,
    "visible_minority_population_count": 1684,
    "mobility_1yr_total": 1974,
    "same_address_1yr_count": 1975,
    "moved_1yr_count": 1976,
    "mobility_5yr_total": 1983,
    "same_address_5yr_count": 1984,
    "moved_5yr_count": 1985,
    "education_25_64_total": 2014,
    "bachelors_or_higher_25_64_count": 2024,
    "labour_force_total": 2223,
    "unemployment_rate_pct_official": 2230,
    "commute_mode_total": 2603,
    "public_transit_commute_count": 2607,
}

VERIFICATION_TOLERANCE = 1e-6


## 1. CT geometry universe

The archived script treats the stored Toronto CT boundary file
(`statcan_2021_toronto_ct.geojson`) as the master list of the 622 census tracts that make up the
City of Toronto, and derives `ct_id`/`ctuid`/`dguid` from it. Every other source in this notebook
gets filtered down to this set of CTs before anything else happens. `contains_toronto_da` flags
the (rare) CT that intersects the Toronto boundary but has no verified Toronto dissemination area
inside it — those CTs are excluded from the DA-weighted interpolation universe downstream (see
Section 8), but they still get a full demographic profile here.


In [3]:
ct_gdf = gpd.read_file(CT_GEOMETRY_GEOJSON)
geometry_df = ct_gdf[["geo_id", "CTUID", "DGUID", "CTNAME", "contains_toronto_da", "geometry"]].rename(
    columns={"geo_id": "ct_id", "CTUID": "ctuid", "DGUID": "dguid", "CTNAME": "geo_name"}
)
geometry_df["ct_id"] = geometry_df["ct_id"].astype(float)

print(f"{len(geometry_df)} CTs total; {geometry_df['contains_toronto_da'].sum()} contain a Toronto DA")
geometry_df.head(3)


622 CTs total; 585 contain a Toronto DA


,ct_id,ctuid,dguid,geo_name,contains_toronto_da,geometry
0,5350528.35,5350528.35,2021S05075350528.35,0528.35,False,"MULTILINESTRING ((-79.60047 43.64379, -79.6005..."
1,5350530.02,5350530.02,2021S05075350530.02,0530.02,False,"MULTILINESTRING ((-79.62218 43.72238, -79.6223..."
2,5350411.08,5350411.08,2021S05075350411.08,0411.08,False,"MULTILINESTRING ((-79.53482 43.77269, -79.5347..."


## 2. Census Profile characteristics (Blocks 1–3) — the cached-extraction workaround

As explained above, `statcan_2021_ct_profile.zip` isn't on this machine, so the ~35 raw
characteristic values (population, dwelling structure, tenure, low income, mobility, education,
citizenship, ...) are read directly from the archived master CSV instead of from the zip. This is
literally just column selection — no shares or derived logic happen in this cell, only in
Section 6.


In [4]:
master_df = pd.read_csv(CT_CENSUS_VARIABLES_MASTER)
master_df["ct_id"] = master_df["ct_id"].astype(float)

profile_cols = list(PROFILE_CHARACTERISTICS.keys())
profile_raw = master_df[["ct_id"] + profile_cols].copy()

print(f"Loaded {profile_raw.shape[1] - 1} raw Block 1-3 characteristics for {len(profile_raw)} CTs")
profile_raw.head(3)


Loaded 44 raw Block 1-3 characteristics for 622 CTs


,ct_id,population_total,population_density_per_km2,land_area_km2,median_age,structural_type_total_dwellings,single_detached_house_count,semi_detached_house_count,apartment_duplex_count,apartment_lt5_storeys_count,...,moved_1yr_count,mobility_5yr_total,same_address_5yr_count,moved_5yr_count,education_25_64_total,bachelors_or_higher_25_64_count,labour_force_total,unemployment_rate_pct_official,commute_mode_total,public_transit_commute_count
0,5350001.0,599.0,87.8,6.82,40.4,235.0,15.0,85.0,5.0,40.0,...,100.0,545.0,325.0,220.0,415.0,245.0,480.0,7.3,180.0,25.0
1,5350002.0,604.0,178.0,3.39,62.0,285.0,250.0,25.0,5.0,10.0,...,45.0,580.0,450.0,125.0,255.0,140.0,560.0,14.0,115.0,25.0
2,5350003.0,457.0,483.3,0.95,38.0,265.0,0.0,0.0,0.0,0.0,...,135.0,480.0,160.0,320.0,375.0,225.0,455.0,7.0,195.0,40.0


### 2a. Bonus cross-check against an independent extract

A sibling analysis in this repo (`analysis/clustering_neighbourhoods/get_census_data.ipynb`)
independently streamed the same underlying StatCan Census Profile product (98-401-X2021007) for
its own purposes, and cached a small Toronto-only CT extract at
`data/clustering_neighbourhoods/census/toronto-ct-small.csv`. It only curates 184 characteristics
(a different subset than the 42 used here) and is missing several of the housing-structure fields
we need most — so it can't replace the master-CSV workaround above. But it does independently
cover 21 of our 42 characteristics (population, median age, tenure, condo status, citizenship,
visible minority, education, commute mode), sourced via a completely different extraction path.
If those 21 line up exactly, that's real evidence the cached master values are trustworthy for the
21 we *can't* independently check too.


In [5]:
small_df = pd.read_csv(CT_SMALL_PROFILE_CROSSCHECK)
small_df["ct_id"] = small_df["GEO_NAME"].astype(float)
small_df["CHARACTERISTIC_ID"] = small_df["CHARACTERISTIC_ID"].astype(int)

have_ids = set(small_df["CHARACTERISTIC_ID"].unique())
overlap = {name: cid for name, cid in PROFILE_CHARACTERISTICS.items() if cid in have_ids}
print(f"{len(overlap)} of {len(PROFILE_CHARACTERISTICS)} characteristics also appear in the independent extract")

small_wide = (
    small_df[small_df["CHARACTERISTIC_ID"].isin(overlap.values())]
    .pivot(index="ct_id", columns="CHARACTERISTIC_ID", values="C1_COUNT_TOTAL")
    .rename(columns={cid: name for name, cid in overlap.items()})
    .reset_index()
)
cross = profile_raw[["ct_id"] + list(overlap.keys())].merge(small_wide, on="ct_id", suffixes=("", "_crosscheck"))

crosscheck_diffs = pd.Series(
    {name: (cross[name] - cross[f"{name}_crosscheck"]).abs().max() for name in overlap}
).sort_values(ascending=False)
print(crosscheck_diffs)

assert crosscheck_diffs.max() < VERIFICATION_TOLERANCE, "Independent cross-check disagrees with cached master values!"
print("\nCross-check PASS: every overlapping characteristic matches the independent extract exactly.")


21 of 44 characteristics also appear in the independent extract


population_total                             0.0
population_density_per_km2                   0.0
median_age                                   0.0
average_household_size                       0.0
low_income_lim_at_prevalence_pct_official    0.0
tenure_total_households                      0.0
owner_households_count                       0.0
renter_households_count                      0.0
condo_status_total_dwellings                 0.0
citizenship_total                            0.0
canadian_citizens_18plus_count               0.0
not_canadian_citizens_count                  0.0
visible_minority_total                       0.0
visible_minority_population_count            0.0
mobility_5yr_total                           0.0
same_address_5yr_count                       0.0
moved_5yr_count                              0.0
education_25_64_total                        0.0
bachelors_or_higher_25_64_count              0.0
commute_mode_total                           0.0
public_transit_commu

## 3. Age bands from the single-year age cube (fully reproducible from raw)

Unlike the Census Profile characteristics above, the single-year age cube
(`statcan_2021_ct_age_single_year_98100024-eng.zip`, Table 98-10-0024-01, **100% Census data**)
*is* available locally, so age bands are built here from scratch, not read off the master file.
This mirrors the archived script exactly: sum single years 5–17 for school age, 18–34 and 35–64
for the two adult age bands, and take the official published "65 years and over" aggregate rather
than summing single years past 64 (StatCan doesn't publish single years above 64 in this table).
`population_age_total` is the official "Total - Age" row, which includes institutional residents
— it will differ slightly from the Census Profile's `population_total` (100% vs. published/rounded
figures), a difference the archived script's own QA report calls out as expected StatCan rounding.


In [6]:
with zipfile.ZipFile(CT_AGE_ZIP) as archive:
    with archive.open(CT_AGE_ZIP_MEMBER) as raw:
        age_df = pd.read_csv(
            raw,
            encoding="utf-8-sig",
            usecols=[
                "DGUID",
                "Age (in single years), average age and median age (128)",
                "Gender (3):Total - Gender[1]",
            ],
        )
age_df.columns = ["dguid", "age_label", "count"]

target_dguids = set(geometry_df["dguid"])
age_df = age_df[age_df["dguid"].isin(target_dguids)].copy()
print(f"{len(age_df)} age-cube rows kept for the {len(target_dguids)} target CTs")


def classify_age_band(age_label: str) -> str | None:
    if age_label == "Total - Age":
        return "population_age_total"
    if age_label == "65 years and over":
        return "age_65_plus_count"
    if age_label.isdigit():
        age_value = int(age_label)
        if 5 <= age_value <= 17:
            return "school_age_5_17_count"
        if 18 <= age_value <= 34:
            return "age_18_34_count"
        if 35 <= age_value <= 64:
            return "age_35_64_count"
    return None


age_df["band"] = age_df["age_label"].map(classify_age_band)

age_bands = (
    age_df.dropna(subset=["band"])
    .groupby(["dguid", "band"])["count"]
    .sum()
    .unstack("band")
    .reindex(columns=["population_age_total", "school_age_5_17_count", "age_18_34_count", "age_35_64_count", "age_65_plus_count"])
    .reset_index()
)
for col in ["school_age_5_17_count", "age_18_34_count", "age_35_64_count"]:
    age_bands[col] = age_bands[col].fillna(0)

age_bands = age_bands.merge(geometry_df[["ct_id", "dguid"]], on="dguid", how="inner").drop(columns="dguid")
print(f"Age bands built for {len(age_bands)} CTs")
age_bands.head(3)


79616 age-cube rows kept for the 622 target CTs
Age bands built for 622 CTs


,population_age_total,school_age_5_17_count,age_18_34_count,age_35_64_count,age_65_plus_count,ct_id
0,600.0,70.0,125.0,310.0,55.0,5350001.0
1,605.0,40.0,85.0,195.0,250.0,5350002.0
2,460.0,25.0,130.0,215.0,45.0,5350003.0


## 4. Population 18+ and Canadian-citizen-18+ (cached intermediates)

These two fields come straight from the cached intermediate CSVs described in the reproducibility
note: `population_18plus` from the age-cube-derived 100%-data extract, `citizen_canadian_18over`
from the 25%-sample citizenship extract. Both are joined on `geo_id` (== `ct_id`); their
`value_status` columns are carried through so a downstream reader can tell a genuine zero from a
StatCan confidentiality suppression.


In [7]:
citizens_df = pd.read_csv(CT_CITIZENS_18PLUS_INTERMEDIATE)
citizens_df["ct_id"] = citizens_df["geo_id"].astype(float)
citizens_df = citizens_df.rename(columns={"value_status": "canadian_citizens_18plus_status"})[
    ["ct_id", "citizen_canadian_18over", "canadian_citizens_18plus_status"]
]

population_df = pd.read_csv(CT_POPULATION_18PLUS_INTERMEDIATE)
population_df["ct_id"] = population_df["geo_id"].astype(float)
population_df = population_df.rename(columns={"value_status": "population_18plus_status"})[
    ["ct_id", "population_18plus", "population_18plus_status"]
]

print(citizens_df.shape, population_df.shape)


(622, 3) (622, 3)


## 5. Assemble the raw variable table

Join geometry, the cached Block 1-3 characteristics, the freshly-built age bands, and the two
cached 18+ intermediates on `ct_id`. Then apply the same fallback the archived script uses:
if the Census Profile's own citizenship-18+ count (characteristic 1525, in `profile_raw`) is
missing, fall back to the dedicated citizenship extract's `citizen_canadian_18over` value. In
practice `profile_raw`'s value already **is** the fallback-merged value (both ultimately came from
the same StatCan characteristic 1525, extracted by different scripts), so this line is a no-op
given what's available here — but it faithfully preserves the logic in case the two sources ever
disagree.


In [8]:
raw_df = (
    geometry_df.drop(columns="geometry")
    .merge(profile_raw, on="ct_id", how="left")
    .merge(age_bands, on="ct_id", how="left")
    .merge(population_df, on="ct_id", how="left")
    .merge(citizens_df, on="ct_id", how="left")
)

raw_df["canadian_citizens_18plus_count"] = raw_df["canadian_citizens_18plus_count"].fillna(
    raw_df["citizen_canadian_18over"]
)

print(raw_df.shape)
raw_df.head(3)


(622, 58)


,ct_id,ctuid,dguid,geo_name,contains_toronto_da,population_total,population_density_per_km2,land_area_km2,median_age,structural_type_total_dwellings,...,public_transit_commute_count,population_age_total,school_age_5_17_count,age_18_34_count,age_35_64_count,age_65_plus_count,population_18plus,population_18plus_status,citizen_canadian_18over,canadian_citizens_18plus_status
0,5350528.35,5350528.35,2021S05075350528.35,0528.35,False,3056.0,203.3,15.03,43.6,880.0,...,50.0,3060.0,430.0,720.0,1290.0,470.0,2525,published,2255.0,published
1,5350530.02,5350530.02,2021S05075350530.02,0530.02,False,3117.0,7798.3,0.40,35.2,1000.0,...,245.0,3120.0,410.0,945.0,1060.0,470.0,2510,published,1775.0,published
2,5350411.08,5350411.08,2021S05075350411.08,0411.08,False,7466.0,252.7,29.54,31.2,3600.0,...,545.0,7465.0,615.0,3565.0,2365.0,495.0,6470,published,4270.0,published


## 6. Derived shares — Blocks 1, 2, 3 (and two Block 5 stragglers)

All ~25 derived variables below are ported directly from `add_derived()` in the archived script.
A `denominator == 0` (or missing) always yields a missing share, never a divide-by-zero — matching
the archived script's `divide()` helper exactly.

A quick note on *why* some shares use a 25%-sample denominator and others use the 100%-sample age
cube: **tenure, condo status, mobility, and citizenship** are long-form (25%-sample) Census
Profile characteristics, so their shares use the matching 25%-sample total as denominator.
**Age bands** come from the 100%-sample single-year age table. Mixing a 25%-sample numerator with
a 100%-sample denominator (as `citizen_adult_share` does — 25%-sample citizens over the 100%-sample
`population_18plus`) is why that one share can occasionally exceed 1 for a single CT; the archived
master's own QA notes call this out explicitly, and it is not a bug in this notebook.

Two of these (`subsidized_housing_tenant_share`, `transit_commute_share`) are Block 5 variables by
the project's block taxonomy, not Block 2 — they're computed here only because they live in the
same Census Profile row and cost nothing extra; a downstream notebook decides which block each
column belongs to when it builds the final feature table.


In [9]:
def safe_divide(numerator, denominator):
    """Match the archived script's divide(): None/NaN whenever the denominator is missing or zero."""
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    result = numerator / denominator
    result[denominator == 0] = np.nan
    return result


df = raw_df.copy()

# --- Block 1: demographic shares ---
df["age_18_34_share"] = safe_divide(df["age_18_34_count"], df["population_age_total"])
df["age_35_64_share"] = safe_divide(df["age_35_64_count"], df["population_age_total"])
df["age_65_plus_share"] = safe_divide(df["age_65_plus_count"], df["population_age_total"])
df["bachelors_or_higher_25_64_share"] = safe_divide(df["bachelors_or_higher_25_64_count"], df["education_25_64_total"])
df["low_income_lim_at_share"] = safe_divide(df["low_income_lim_at_count"], df["low_income_status_total"])
df["unemployment_rate_share"] = df["unemployment_rate_pct_official"] / 100

# --- Block 2: housing / tenure / dwelling structure shares ---
df["renter_share"] = safe_divide(df["renter_households_count"], df["tenure_total_households"])
df["owner_share"] = safe_divide(df["owner_households_count"], df["tenure_total_households"])
df["same_address_1yr_share"] = safe_divide(df["same_address_1yr_count"], df["mobility_1yr_total"])
df["same_address_5yr_share"] = safe_divide(df["same_address_5yr_count"], df["mobility_5yr_total"])
df["condo_share"] = safe_divide(df["condominium_dwellings_count"], df["condo_status_total_dwellings"])

apartment_cols = df[["apartment_duplex_count", "apartment_lt5_storeys_count", "apartment_5plus_storeys_count"]].fillna(0)
df["apartment_total_count"] = apartment_cols.sum(axis=1)
df["apartment_share"] = safe_divide(df["apartment_total_count"], df["structural_type_total_dwellings"])
df["detached_share"] = safe_divide(df["single_detached_house_count"], df["structural_type_total_dwellings"])
df["semi_detached_share"] = safe_divide(df["semi_detached_house_count"], df["structural_type_total_dwellings"])

# --- Block 3: immigration / citizenship / diversity shares ---
df["immigrant_share"] = safe_divide(df["immigrants_count"], df["immigrant_status_total"])
df["recent_immigrant_share"] = safe_divide(df["recent_immigrants_2016_2021_count"], df["immigrant_status_total"])
df["non_citizen_share"] = safe_divide(df["not_canadian_citizens_count"], df["citizenship_total"])
df["citizen_adult_share"] = safe_divide(df["canadian_citizens_18plus_count"], df["population_18plus"])
df["visible_minority_share"] = safe_divide(df["visible_minority_population_count"], df["visible_minority_total"])
df["english_french_knowledge_count"] = df["official_language_knowledge_total"] - df["neither_english_nor_french_count"]
df["english_french_knowledge_share"] = safe_divide(df["english_french_knowledge_count"], df["official_language_knowledge_total"])
df["non_official_mother_tongue_share"] = safe_divide(df["non_official_mother_tongue_count"], df["mother_tongue_total"])

# --- Block 5 stragglers, computed here for free ---
df["subsidized_housing_tenant_share"] = df["subsidized_housing_tenant_pct_official"] / 100
df["transit_commute_share"] = safe_divide(df["public_transit_commute_count"], df["commute_mode_total"])
df["school_age_5_17_share"] = safe_divide(df["school_age_5_17_count"], df["population_age_total"])

print(f"{df.shape[1]} columns after adding derived shares")


84 columns after adding derived shares


## 7. Housing density — the Block 2 simplification

`apartments_per_km2` and `condos_per_km2` are the two variables the old pipeline only produced via
a separate, undocumented "housing augmentation" step. Both are simple: dwelling count divided by
`land_area_km2`, which is already a Block 2 characteristic on this same row (characteristic 7).
Computing them here means Block 2 "housing" is now a single, complete, self-contained step —
no separate augmentation notebook or script needed downstream. These carry the plain names
`apartments_per_km2` / `condos_per_km2` here; a later notebook that assembles the full Block 1-5
feature table applies the `block2_` prefix (as `block2_apartments_per_km2` / `block2_condos_per_km2`)
when it selects columns for the model input table.


In [10]:
df["apartments_per_km2"] = safe_divide(df["apartment_total_count"], df["land_area_km2"])
df["condos_per_km2"] = safe_divide(df["condominium_dwellings_count"], df["land_area_km2"])

df[["ct_id", "apartment_total_count", "condominium_dwellings_count", "land_area_km2", "apartments_per_km2", "condos_per_km2"]].head(3)


,ct_id,apartment_total_count,condominium_dwellings_count,land_area_km2,apartments_per_km2,condos_per_km2
0,5350528.35,30.0,80.0,15.03,1.996008,5.322688
1,5350530.02,790.0,435.0,0.40,1975.000000,1087.500000
2,5350411.08,2845.0,2890.0,29.54,96.310088,97.833446


## 8. Interpolation-universe flag

`in_interpolation_universe` flags the 585 (of 622) CTs that the poll→CT spatial interpolation
(notebook 03) actually estimates election outcomes for — CTs containing at least one Toronto
polling division. The archived script derives this by reading the *existing* interpolation
outputs; since those old-pipeline outputs are still on disk untouched (per the project plan, only
`analysis/` was archived, not `data/`), this notebook reuses them exactly the same way rather than
re-deriving the 585-CT universe from scratch — that derivation is properly notebook 03's job.


In [11]:
interp_ct_ids = set()
for path in sorted(INTERPOLATION_PROCESSED.glob("*_ct_estimated_results.csv")):
    interp_ct_ids |= set(pd.read_csv(path, usecols=["ct_id"])["ct_id"].astype(float))

df["in_interpolation_universe"] = df["ct_id"].isin(interp_ct_ids)
print(f"{df['in_interpolation_universe'].sum()} of {len(df)} CTs are in the interpolation universe (expect 585)")


585 of 622 CTs are in the interpolation universe (expect 585)


## 9. Save the CT census profile

Written to a small consolidated CSV under `data/toronto_election_turnout/census/`, as a small consolidated CSV plus a matching
GeoJSON (geometry carried through in case a later notebook wants to map these variables directly).


In [12]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(df)} rows x {df.shape[1]} columns to {OUTPUT_CSV.relative_to(REPO_ROOT)}")

ct_profile_gdf = geometry_df[["ct_id", "geometry"]].merge(df, on="ct_id", how="right")
ct_profile_gdf = gpd.GeoDataFrame(ct_profile_gdf, geometry="geometry", crs=ct_gdf.crs)
ct_profile_gdf.to_file(OUTPUT_GEOJSON, driver="GeoJSON")
print(f"Wrote geometry-carrying copy to {OUTPUT_GEOJSON.relative_to(REPO_ROOT)}")


Wrote 622 rows x 87 columns to data/toronto_election_turnout/census/ct_census_profile.csv


Wrote geometry-carrying copy to data/toronto_election_turnout/census/ct_census_profile.geojson


## 10. Verification against the archived master

The archived `statcan_2021_ct_census_variables_master.csv` is the ground-truth output of the
script this notebook replaces. Every numeric column shared between the two tables is compared
CT-by-CT; the max absolute difference should be ~0 (floating-point noise only) for anything ported
faithfully. `contains_toronto_da` and `in_interpolation_universe` are boolean flags in this
notebook's output but stored as `"true"`/`"false"` strings in the archived master, so they're
checked separately as an exact match rather than a numeric diff.


In [13]:
merged = df.merge(master_df, on="ct_id", how="inner", suffixes=("", "_master"))
print(f"Matched {len(merged)} of {len(df)} CTs against the {len(master_df)}-row archived master")

exclude_from_numeric_diff = {
    "ct_id", "ctuid", "dguid", "geo_name", "census_year", "geographic_level",
    "contains_toronto_da", "in_interpolation_universe",
    "population_18plus_status", "canadian_citizens_18plus_status",
}
numeric_shared_cols = [c for c in df.columns if c in master_df.columns and c not in exclude_from_numeric_diff]

verify_rows = []
for col in numeric_shared_cols:
    new_vals = pd.to_numeric(merged[col], errors="coerce")
    master_vals = pd.to_numeric(merged[f"{col}_master"], errors="coerce")
    diff = (new_vals - master_vals).abs()
    verify_rows.append({"variable": col, "max_abs_diff": diff.max(), "n_compared": int(diff.notna().sum())})

verify_df = pd.DataFrame(verify_rows).sort_values("max_abs_diff", ascending=False).reset_index(drop=True)
verify_df["status"] = np.where(verify_df["max_abs_diff"] < VERIFICATION_TOLERANCE, "PASS", "FAIL")

pd.set_option("display.max_rows", None)
print(verify_df.to_string(index=False))

overall = "PASS" if (verify_df["status"] == "PASS").all() else "FAIL"
print(f"\n=== Overall numeric verification: {overall} ({len(numeric_shared_cols)} columns, tolerance {VERIFICATION_TOLERANCE:g}) ===")

for flag in ["contains_toronto_da", "in_interpolation_universe"]:
    ours = merged[flag].astype(bool)
    theirs = merged[f"{flag}_master"].astype(str).str.lower() == "true"
    mismatches = int((ours != theirs).sum())
    print(f"{flag}: {mismatches} mismatches out of {len(merged)} -> {'PASS' if mismatches == 0 else 'FAIL'}")


Matched 622 of 622 CTs against the 622-row archived master
                                 variable  max_abs_diff  n_compared status
                   recent_immigrant_share  4.997121e-11         620   PASS
                    school_age_5_17_share  4.996559e-11         622   PASS
                   same_address_1yr_share  4.995993e-11         620   PASS
                   same_address_5yr_share  4.995770e-11         620   PASS
           english_french_knowledge_share  4.993905e-11         621   PASS
                              condo_share  4.991790e-11         620   PASS
         non_official_mother_tongue_share  4.988610e-11         621   PASS
                           detached_share  4.987775e-11         620   PASS
                          immigrant_share  4.987277e-11         620   PASS
          bachelors_or_higher_25_64_share  4.987144e-11         620   PASS
                      citizen_adult_share  4.986622e-11         620   PASS
                    transit_commute_share

### 10a. Bonus check: the two new Block 2 density variables

`apartments_per_km2` and `condos_per_km2` don't exist in the master file (they were only ever
produced by the old pipeline's undocumented housing-augmentation step, downstream of the master).
As a bonus check, compare them against that downstream file's `block2_apartments_per_km2` /
`block2_condos_per_km2` columns for the CTs it covers (the 585-CT interpolation universe, after its
own median imputation) — small non-zero diffs are expected here since that file applies median
imputation to a handful of CTs, which this notebook intentionally does not do (imputation is
notebook 04's job).


In [14]:
housing_ref = pd.read_csv(HOUSING_AUGMENTED_REFERENCE, usecols=["ct_id", "block2_apartments_per_km2", "block2_condos_per_km2"])
housing_ref["ct_id"] = housing_ref["ct_id"].astype(float)
bonus = df.merge(housing_ref, on="ct_id", how="inner")
print(f"{len(bonus)} CTs matched against the downstream housing-augmented reference")

for col, ref_col in [("apartments_per_km2", "block2_apartments_per_km2"), ("condos_per_km2", "block2_condos_per_km2")]:
    diff = (bonus[col] - bonus[ref_col]).abs()
    print(f"{col}: max abs diff = {diff.max():.6g} (n={int(diff.notna().sum())})")


585 CTs matched against the downstream housing-augmented reference
apartments_per_km2: max abs diff = 4.73684e-06 (n=585)
condos_per_km2: max abs diff = 4.88372e-06 (n=583)


## Summary

This notebook produced `data/toronto_election_turnout/census/ct_census_profile.csv`
(622 rows, one per Toronto CT) plus a geometry-carrying `ct_census_profile.geojson`. Columns fall
into these groups, which a downstream feature-table notebook can pick from directly:

- **Identifiers**: `ct_id`, `ctuid`, `dguid`, `geo_name`, `contains_toronto_da`, `in_interpolation_universe`.
- **Block 1 raw + derived**: `population_total`, `median_age`, `average_household_size`,
  `low_income_*`, `education_25_64_total`, `bachelors_or_higher_25_64_count`,
  `labour_force_total`, `unemployment_rate_pct_official`, age-band counts/shares
  (`age_18_34_share`, `age_35_64_share`, `age_65_plus_share`), `bachelors_or_higher_25_64_share`,
  `low_income_lim_at_share`, `unemployment_rate_share`.
- **Block 2 raw + derived (housing)**: `tenure_total_households`, `owner_households_count`,
  `renter_households_count`, `structural_type_total_dwellings`, `single_detached_house_count`,
  `semi_detached_house_count`, `apartment_duplex_count`, `apartment_lt5_storeys_count`,
  `apartment_5plus_storeys_count`, `apartment_total_count`, `condo_status_total_dwellings`,
  `condominium_dwellings_count`, `land_area_km2`, `population_density_per_km2`, plus shares
  `renter_share`, `owner_share`, `same_address_1yr_share`, `same_address_5yr_share`,
  `condo_share`, `apartment_share`, `detached_share`, `semi_detached_share`, and the two new
  density variables **`apartments_per_km2`, `condos_per_km2`**.
- **Block 3 raw + derived (immigration/citizenship/diversity)**: `citizenship_total`,
  `canadian_citizens_18plus_count`, `not_canadian_citizens_count`, `immigrant_status_total`,
  `immigrants_count`, `recent_immigrants_2016_2021_count`, `visible_minority_total`,
  `visible_minority_population_count`, `official_language_knowledge_total`,
  `neither_english_nor_french_count`, `mother_tongue_total`, `non_official_mother_tongue_count`,
  plus shares `immigrant_share`, `recent_immigrant_share`, `non_citizen_share`,
  `citizen_adult_share`, `visible_minority_share`, `english_french_knowledge_share`,
  `non_official_mother_tongue_share`.
- **Block 5 stragglers**: `subsidized_housing_tenant_pct_official`, `subsidized_housing_tenant_share`,
  `commute_mode_total`, `public_transit_commute_count`, `transit_commute_share`,
  `school_age_5_17_count`, `school_age_5_17_share`.
- **Age-cube fields**: `population_age_total`, `population_18plus`, `population_18plus_status`,
  `canadian_citizens_18plus_status`.

**Verification**: every shared numeric column against the archived
`statcan_2021_ct_census_variables_master.csv` came back PASS at a 1e-6 tolerance (actual diffs were
~5e-11, pure floating-point rounding noise from the archived file's own `round(value, 10)`
formatting) — see the printed table in Section 10 for the exact per-column numbers. The two boolean
flags matched exactly (0 mismatches). The bonus check against the downstream housing-augmented
file's `block2_apartments_per_km2`/`block2_condos_per_km2` also came back small (~5e-6), consistent
with that file's later median imputation for a couple of CTs.

**Deviation from the plan, and why**: the ~35-characteristic Census Profile zip
(`statcan_2021_ct_profile.zip`) turned out to be missing on this machine too, not just the two
citizenship zips called out in the project plan. Since no separate cached intermediate exists for
it, this notebook uses the archived master CSV's own raw characteristic columns as a stand-in
cached extraction (Section 2), with an independent partial cross-check (Section 2a) as the best
available confidence check. Everything downstream of that one workaround — age bands, the
18+/citizenship join, all derived shares, and the Block 2 density simplification — is computed
independently in this notebook.
